In [52]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

nama_file = "URL dataset.csv" 
print("Sedang memuat dataset besar...")
df = pd.read_csv(nama_file)

print(f"Total data yang dimuat: {len(df)}")
print("Komposisi label:\n", df['type'].value_counts())

# Melakukan label encoding
df['type_encoded'] = df['type'].map({'phishing': 1, 'legitimate': 0}).fillna(0)

# Split 80% Train, 20% Test 
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)


# 2. PREPROCESSING
print("\nMelakukan tokenisasi karakter (proses ini butuh waktu beberapa saat)...")
max_chars = 200

tokenizer = Tokenizer(char_level=True, lower=True)
tokenizer.fit_on_texts(df_train['url'].values)

# Transformasi teks ke angka
X_train = pad_sequences(tokenizer.texts_to_sequences(df_train['url'].values), maxlen=max_chars, padding='post')
y_train = df_train['type_encoded'].values.astype('float32')

X_test = pad_sequences(tokenizer.texts_to_sequences(df_test['url'].values), maxlen=max_chars, padding='post')
y_test = df_test['type_encoded'].values.astype('float32')

num_classes = len(tokenizer.word_index) + 1

# 3. Model CNN
# Desain ini disesuaikan untuk kapasitas data yang besar agar tidak gampang underfitting
model = Sequential([
    Embedding(input_dim=num_classes, output_dim=64, input_length=max_chars),
    
    Conv1D(filters=128, kernel_size=7, activation='relu'),
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.5), # Mencegah overfitting pada data raksasa
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. OPTIMASI TRAINING (BATCH SIZE BESAR & CALLBACKS)
# Berhenti otomatis jika model sudah tidak berkembang
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print("\n--- Memulai Proses Training 500rb Data ---")
# batch_size dinaikkan ke 512 agar proses training melesat cepat
model.fit(
    X_train, y_train, 
    epochs=15, 
    batch_size=512, 
    validation_split=0.2, # 10% dari data train untuk validasi berkala
    callbacks=[early_stopping]
)


print("\n--- Memulai Proses Pengujian ---")
prediksi_probabilitas = model.predict(X_test)
y_pred = (prediksi_probabilitas > 0.5).astype("int32")

# Membuat tabel hasil asli dari data test Anda
tabel_hasil = pd.DataFrame({
    'URL': df_test['url'],
    'Status Asli': df_test['type'],
    'Prediksi Model': ['phishing' if p == 1 else 'legitimate' for p in y_pred.flatten()],
    'Probabilitas Phishing': prediksi_probabilitas.flatten()
})

tabel_hasil['Kesimpulan'] = np.where(
    tabel_hasil['Status Asli'] == tabel_hasil['Prediksi Model'], 
    '✅ BENAR', 
    '❌ SALAH'
)

total_benar = (tabel_hasil['Kesimpulan'] == '✅ BENAR').sum()
print("\n" + "="*40)
print(f"Total Tebakan Benar: {total_benar} dari {len(df_test)} data test")
print(f"Akurasi Akhir Model: {(total_benar/len(df_test))*100:.2f}%")
print("="*40)

Sedang memuat dataset besar...
Total data yang dimuat: 450176
Komposisi label:
 type
legitimate    345738
phishing      104438
Name: count, dtype: int64

Melakukan tokenisasi karakter (proses ini butuh waktu beberapa saat)...

--- Memulai Proses Training 500rb Data ---
Epoch 1/15


c:\Users\skywe\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


563/563 ━━━━━━━━━━━━━━━━━━━━ 136s 239ms/step - accuracy: 0.9861 - loss: 0.0391 - val_accuracy: 0.9963 - val_loss: 0.0158
Epoch 2/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 145s 245ms/step - accuracy: 0.9966 - loss: 0.0152 - val_accuracy: 0.9972 - val_loss: 0.0123
Epoch 3/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 142s 251ms/step - accuracy: 0.9972 - loss: 0.0119 - val_accuracy: 0.9972 - val_loss: 0.0107
Epoch 4/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 137s 243ms/step - accuracy: 0.9978 - loss: 0.0098 - val_accuracy: 0.9978 - val_loss: 0.0093
Epoch 5/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 131s 232ms/step - accuracy: 0.9980 - loss: 0.0080 - val_accuracy: 0.9979 - val_loss: 0.0088
Epoch 6/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 136s 242ms/step - accuracy: 0.9984 - loss: 0.0066 - val_accuracy: 0.9978 - val_loss: 0.0097
Epoch 7/15
563/563 ━━━━━━━━━━━━━━━━━━━━ 146s 258ms/step - accuracy: 0.9986 - loss: 0.0053 - val_accuracy: 0.9980 - val_loss: 0.0090

--- Memulai Proses Pengujian ---
2814/2814 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step

Total Te

In [53]:
model.save("deteks_phishing_cnn.keras")

In [54]:
import pickle

# Menentukan nama file untuk menyimpan tokenizer
nama_file_tokenizer = 'tokenizer_phishing.pickle'

# Proses mengekspor objek tokenizer ke dalam file biner
with open(nama_file_tokenizer, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"✅ Tokenizer berhasil diekspor dan disimpan dengan nama: {nama_file_tokenizer}")

✅ Tokenizer berhasil diekspor dan disimpan dengan nama: tokenizer_phishing.pickle


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("Memuat model dan tokenizer lama...")
model = load_model("deteks_phishing_cnn.keras")

with open('tokenizer_phishing.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

# 2. LOAD DATASET PHISHING BARU
print("Memuat dataset baru...")
# Ganti dengan nama file dataset baru 
file_baru = "Phishing URLs.csv" 
df_baru = pd.read_csv(file_baru)

# Pastikan label di-encode menjadi angka (phishing: 1, legitimate: 0)
df_baru['type_encoded'] = df_baru['Type'].map({'phishing': 1}).fillna(0)

# 3. PREPROCESSING DATASET
max_chars = 200 

sequences_baru = tokenizer.texts_to_sequences(df_baru['url'].values)
X_new = pad_sequences(sequences_baru, maxlen=max_chars, padding='post')
y_new = df_baru['type_encoded'].values.astype('float32')


# 4. SETTING OPTIMIZER (LOWER LEARNING RATE)
# Kita compile ulang model dengan learning rate lebih rendah (misal: 0.0001)
# Ini agar model beradaptasi perlahan dengan data baru tanpa merusak ingatan lama
model.compile(
    optimizer=Adam(learning_rate=0.0001), 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

# 5. PROSES TRAINING ULANG (RE-TRAINING)
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

print("\n--- Memulai Proses Melatih Ulang Model ---")
model.fit(
    X_new, y_new,
    epochs=5, # Tidak perlu epoch terlalu banyak karena model sudah pintar
    batch_size=512,
    validation_split=0.2, # 20% data baru untuk validasi
    callbacks=[early_stopping]
)

# 6. SIMPAN MODEL VERSI TERBARU (V2)
# Simpan dengan nama baru atau timpa model lama
model.save("model_phishing_cnn_v2.keras")
print("\n✅ Model berhasil diperbarui dan disimpan sebagai 'model_phishing_cnn_v2.keras'!")

Memuat model dan tokenizer lama...
Memuat dataset baru...

--- Memulai Proses Melatih Ulang Model ---
Epoch 1/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 16s 168ms/step - accuracy: 0.8338 - loss: 0.9321 - val_accuracy: 0.9866 - val_loss: 0.0342
Epoch 2/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 15s 177ms/step - accuracy: 0.9954 - loss: 0.0130 - val_accuracy: 0.9988 - val_loss: 0.0050
Epoch 3/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 16s 188ms/step - accuracy: 0.9990 - loss: 0.0033 - val_accuracy: 0.9995 - val_loss: 0.0016
Epoch 4/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step - accuracy: 0.9996 - loss: 0.0013 - val_accuracy: 0.9998 - val_loss: 8.5363e-04
Epoch 5/5
86/86 ━━━━━━━━━━━━━━━━━━━━ 16s 188ms/step - accuracy: 0.9998 - loss: 8.1556e-04 - val_accuracy: 0.9998 - val_loss: 5.6972e-04

✅ Model berhasil diperbarui dan disimpan sebagai 'model_phishing_cnn_v2.keras'!


In [51]:
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# 1. MUAT MODEL & TOKENIZER
# ==========================================
print("Sedang memuat sistem deteksi AI...")
model = load_model("deteksi_phishing_cnn.keras")

with open('tokenizer_phishing.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)
print("Sistem Siap! Silakan uji URL Anda.\n")

# ==========================================
# 2. INPUT STRING & PREDIKSI
# ==========================================
while True:
    # Meminta input URL dari pengguna
    input_url = input("Masukkan URL yang ingin dicek (ketik 'keluar' untuk berhenti): ")
    
    # Keluar dari program jika mengetik 'keluar'
    if input_url.lower() == 'keluar':
        print("Program selesai. Terima kasih!")
        break
        
    if not input_url.strip():
        print("URL tidak boleh kosong!\n")
        continue

    # Preprocessing: Ubah teks input ke bentuk matriks angka
    # Input dibungkus [input_url] karena tokenizer menerima bentuk list/array
    sequence = tokenizer.texts_to_sequences([input_url])
    X_pred = pad_sequences(sequence, maxlen=200, padding='post')

    # Prediksi menggunakan model CNN
    probabilitas = model.predict(X_pred, verbose=0)[0][0]

    # ==========================================
    # 3. TAMPILKAN HASIL
    # ==========================================
    print("-" * 50)
    if probabilitas > 0.5:
        print(f"HASIL   : ⚠️  PHISHING (Berbahaya!)")
    else:
        print(f"HASIL   : ✅ LEGITIMATE (Aman)")
        
    print(f"AKURASI : {probabilitas * 100:.2f}% indikasi phishing")
    print("-" * 50 + "\n")

Sedang memuat sistem deteksi AI...
Sistem Siap! Silakan uji URL Anda.

--------------------------------------------------
HASIL   : ✅ LEGITIMATE (Aman)
AKURASI : 0.00% indikasi phishing
--------------------------------------------------

--------------------------------------------------
HASIL   : ⚠️  PHISHING (Berbahaya!)
AKURASI : 78.94% indikasi phishing
--------------------------------------------------

--------------------------------------------------
HASIL   : ✅ LEGITIMATE (Aman)
AKURASI : 0.00% indikasi phishing
--------------------------------------------------

Program selesai. Terima kasih!
